In [ ]:
# Import tools for creating the dataset and splitting it later.
from sklearn import datasets, model_selection
# Import plotting tools.
import matplotlib.pyplot as plt
# Import NumPy for later numerical work.
import numpy as np

# Set the number of points to generate.
NUM_DATAPOINTS = 300

# Create two noisy circles. x_data contains each point's two coordinates;
# y_data contains its class label (0 for outside, 1 for inside).
x_data, y_data = datasets.make_circles(n_samples=NUM_DATAPOINTS, noise=0.15, factor=0.6)

# Create a square figure and axes for the plot.
fig, ax = plt.subplots(figsize=(7, 7))
# Plot feature 0 horizontally and feature 1 vertically, coloring points by class.
scatter = ax.scatter(
    x_data[:, 0],
    x_data[:, 1],
    c=y_data,
    edgecolors="grey",
    cmap="Blues",
    vmin=0,
    vmax=1,
)
# Label the axes and identify the two classes in the legend.
ax.set_xlabel("Feature 0")
ax.set_ylabel("Feature 1")
ax.legend(handles=scatter.legend_elements()[0], labels=["outside (0)", "inside (1)"])
# Display the finished plot.
plt.show()


In [ ]:
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)


In [ ]:
from sklearn import linear_model, metrics, svm

linear_clf = linear_model.LogisticRegression()
linear_clf.fit(x_train, y_train)

svm_clf = svm.SVC(gamma=2, C=1, probability=True)
svm_clf.fit(x_train, y_train)

print(linear_clf.score(x_test, y_test), svm_clf.score(x_test, y_test))


In [ ]:
# Feel free to skip over this cell, it's just plotting code.
def plot_decision_surface(clf, xs, ys, ax, title):
    xs = np.linspace(xs[0], xs[1], 100)
    ys = np.linspace(ys[0], ys[1], 100)
    x_mesh, y_mesh = np.meshgrid(xs, ys)

    vis_x = np.stack((x_mesh, y_mesh), axis=2).reshape(-1, 2)
    vis_y = clf.predict_proba(vis_x)[:, 1].reshape(100, 100)

    contour = ax.contourf(
        x_mesh, y_mesh, vis_y, levels=np.linspace(0, 1, 33), cmap="Blues"
    )

    contour.set_clim(0, 1)
    ax.scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        edgecolors="grey",
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    ax.set_title(title)
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")


In [ ]:
from matplotlib import cm

fig, ax = plt.subplots(ncols=2, figsize=(18, 7))
fig.colorbar(cm.ScalarMappable(cmap="Blues"), ticks=np.linspace(0, 1, 11), ax=ax)
plot_decision_surface(linear_clf, (-1.5, 1.5), (-1.5, 1.5), ax[0], "Logistic Regression")
plot_decision_surface(svm_clf, (-1.5, 1.5), (-1.5, 1.5), ax[1], "SVM")


In [ ]:
linear_preds = linear_clf.predict_proba(x_test)[:, 1]
svm_preds = svm_clf.predict_proba(x_test)[:, 1]

linear_curve = metrics.roc_curve(y_test, linear_preds)
svm_curve = metrics.roc_curve(y_test, svm_preds)

plt.plot(linear_curve[0], linear_curve[1], label="Logistic ROC-AUC: %.2f" % metrics.roc_auc_score(y_test, linear_preds))
plt.plot(svm_curve[0], svm_curve[1], label="SVM ROC-AUC: %.2f" % metrics.roc_auc_score(y_test, svm_preds))
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()
